# Teach -> Explain: what happens inside one inference

Teach the model **unseen information**; it must then **explain what it learned** (grounded, not hallucinated). LM **frozen**, knowledge in a **graph**, real **4-bit** LM (~2GB, fits a 6GB GPU).

Run top-to-bottom in molab (repo already cloned).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())   # repo root (where v5/ lives); adjust if cloned elsewhere
from v5.runtime.membrane import TRMRetriever, learn_any, seed_graph, encode_batch
from v5.runtime.dcpd_latent import WhiteBox
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown

wb   = WhiteBox("Qwen/Qwen2.5-3B-Instruct", quant="4bit")   # ~2GB, fits 6GB
g    = seed_graph(); retr = TRMRetriever(g)
def clean(t):
    for c in ("
Human:", "Human:", "
You are", "

", "<|"):
        i = t.find(c);  t = t[:i] if i > 0 else t
    return t.strip()
display(Markdown(f"**LM** `{wb.name}` - quant **{wb.quant}** - VRAM **{wb.vram_gb:.2f}GB** "
                 f"{'fits 6GB' if wb.vram_gb<=6 else 'OVER 6GB'} - seed graph **{len(g)}** atoms"))

## Teach an invented fact the 3B cannot know, then watch it learn + explain
Edit `FACT` / `Q` for your own.

In [ ]:
FACT = "The Klarn Protocol requires exactly three handshake phases: greet, verify, and seal."
Q    = "How many handshake phases does the Klarn Protocol have, and what are they?"

before = clean(wb.generate_plain(f"Answer in one short sentence. {Q}", max_new=40))
display(Markdown(f"### 1 - BEFORE (frozen LM, never saw it)
**Q:** {Q}

LM: {before}"))

n0 = len(g); res = learn_any(g, retr, FACT, name="taught_fact")
display(Markdown(f"### 2 - TEACH via learn_any
node **{res['node']}** - type **{res['kind']}** - graph {n0}->{len(g)}"))

M, order = g.matrix(); q = encode_batch([Q])[0]; sims = (M @ q)
rank = sorted(zip(order, sims.tolist()), key=lambda z: -z[1])[:6]
df = pd.DataFrame(rank, columns=["node", "similarity"]); df["type"] = [g.get(n).kind for n,_ in rank]
display(Markdown("### 3 - RETRIEVE (neural MiniLM ranking)")); display(df)
picked = rank[0][0]; node = g.get(picked)

after = clean(wb.generate_plain(
    f"Use ONLY this learned fact to answer.
Fact: {node.description}
Question: {Q}
Answer:", max_new=40))
display(Markdown(f"### 4 - EXPLAIN (grounded in {picked})
LM+graph: {after}"))
display(Markdown(f"### Before vs After
| | answer |
|---|---|
| LM alone | {before[:70]} |
| LM + graph | {after[:70]} |"))

## The graph after teaching (gold = the just-learned node)

In [ ]:
import networkx as nx
G = nx.DiGraph(); col = {"atom":"#4f8bf5","concept":"#3fb950","procedure":"#d29922","trap":"#f85149"}
for n in g.atoms: G.add_node(n)
for s, d, r in g.edges: G.add_edge(s, d)
pos  = nx.spring_layout(G, seed=3, k=0.9)
cols = ["#ffd23f" if n==res["node"] else col.get(g.get(n).kind, "#888") for n in G.nodes]
sz   = [1500 if n==res["node"] else 700 for n in G.nodes]
plt.figure(figsize=(9,6))
nx.draw(G, pos, node_color=cols, node_size=sz, with_labels=True, font_size=7, edge_color="#ccc")
plt.title("graph after teaching - gold = just-learned node, green = facts, blue = skills"); plt.show()